In [1]:
# from pathlib import Path
# import pandas as pd
# import numpy as np

# try:
#     from IPython.display import display
# except Exception:
#     def display(x):
#         print(x)

# # ============================================================
# # Aggregate all experiment summary spreadsheets
# # Clean full script for notebook/script use
# # ============================================================

# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
# EXPERIMENT_RESULTS_ROOT = OUTPUTS_ROOT / "experiment_results"
# REPORTS_DIR = OUTPUTS_ROOT / "reports"
# REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# BYTES_IN_GB = 1024 ** 3


# # ============================================================
# # 1. Find summary files
# #    Ignore summary_experiment.csv to avoid duplicate rows
# # ============================================================

# summary_files = sorted(
#     f for f in EXPERIMENT_RESULTS_ROOT.glob("**/*summary*.csv")
#     if f.name != "summary_experiment.csv"
# )

# print("Found summary files:")
# for f in summary_files:
#     print("-", f)

# if not summary_files:
#     raise ValueError("No summary files found.")


# # ============================================================
# # 2. Target schema
# # ============================================================

# base_columns = [
#     "timestamp_utc",
#     "experiment_name",
#     "run",
#     "split",
#     "rows",
#     "seconds",
#     "seconds_per_item",
#     "comet",
#     "model_id",
#     "prompt_text",
#     "max_new_tokens",
#     "comet_batch_size",
#     "predictions_csv",
#     "quantization_method",
#     "precision",
#     "eval_split_source",
# ]

# extra_columns = [
#     "original_total_bytes",
#     "compressed_total_bytes",
#     "compression_ratio",
#     "compression_method",
#     "comet_diff_vs_baseline",
#     "speedup_vs_baseline",
#     "current_total_bytes",
#     "size_scope",
#     "model_size_on_disk_gb",
#     "compression_ratio_disk",
# ]

# all_target_columns = base_columns + extra_columns


# # ============================================================
# # 3. Helpers
# # ============================================================

# def first_present(df: pd.DataFrame, candidates, default=np.nan):
#     for c in candidates:
#         if c in df.columns:
#             return df[c]
#     return pd.Series([default] * len(df), index=df.index)


# def infer_precision(df: pd.DataFrame) -> pd.Series:
#     run_series = df["run"].astype(str).str.lower() if "run" in df.columns else pd.Series([""] * len(df), index=df.index)
#     q_series = df["quantization_method"].astype(str).str.lower() if "quantization_method" in df.columns else pd.Series([""] * len(df), index=df.index)

#     precision = pd.Series(["custom"] * len(df), index=df.index)
#     precision[(run_series.str.contains("fp16")) | (q_series.eq("none"))] = "fp16"
#     precision[(run_series.str.contains("int8")) | (q_series.str.contains("int8")) | (q_series.str.contains("bitsandbytes"))] = "int8"
#     return precision


# # ============================================================
# # 4. Normalize one dataframe
# # ============================================================

# def normalize_summary_df(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
#     df = df.copy()
#     df["summary_csv"] = str(file_path)

#     if "timestamp_utc" not in df.columns:
#         df["timestamp_utc"] = pd.NaT

#     if "experiment_name" not in df.columns:
#         df["experiment_name"] = file_path.parent.name

#     if "split" not in df.columns:
#         df["split"] = "eval"

#     if "seconds_per_item" not in df.columns:
#         if "seconds" in df.columns and "rows" in df.columns:
#             df["seconds_per_item"] = df["seconds"] / df["rows"]
#         else:
#             df["seconds_per_item"] = np.nan

#     if "quantization_method" not in df.columns:
#         if "compression_method" in df.columns:
#             df["quantization_method"] = df["compression_method"]
#         else:
#             df["quantization_method"] = "unknown"

#     if "precision" not in df.columns:
#         df["precision"] = infer_precision(df)

#     if "prompt_text" not in df.columns:
#         df["prompt_text"] = np.nan

#     if "comet_batch_size" not in df.columns:
#         df["comet_batch_size"] = np.nan

#     if "predictions_csv" not in df.columns:
#         df["predictions_csv"] = np.nan

#     if "eval_split_source" not in df.columns:
#         df["eval_split_source"] = df["split"]

#     # Unify old and new size field names
#     if "original_total_bytes" not in df.columns:
#         df["original_total_bytes"] = first_present(df, ["original_total_linear_bytes_est", "original_total_bytes"])

#     if "compressed_total_bytes" not in df.columns:
#         df["compressed_total_bytes"] = first_present(df, ["compressed_total_linear_bytes_est", "compressed_total_bytes"])

#     if "current_total_bytes" not in df.columns:
#         df["current_total_bytes"] = first_present(df, ["current_total_linear_bytes_est", "current_total_bytes"])

#     if "compression_ratio" not in df.columns:
#         df["compression_ratio"] = first_present(df, ["compression_ratio_est", "compression_ratio"])

#     if "size_scope" not in df.columns:
#         df["size_scope"] = first_present(df, ["size_scope"], default=np.nan)

#     if "model_size_on_disk_gb" not in df.columns:
#         df["model_size_on_disk_gb"] = first_present(df, ["model_size_on_disk_gb"], default=np.nan)

#     if "compression_ratio_disk" not in df.columns:
#         df["compression_ratio_disk"] = first_present(df, ["compression_ratio_disk"], default=np.nan)

#     for col in all_target_columns:
#         if col not in df.columns:
#             df[col] = np.nan

#     ordered_cols = all_target_columns + ["summary_csv"]
#     other_cols = [c for c in df.columns if c not in ordered_cols]
#     return df[ordered_cols + other_cols]


# # ============================================================
# # 5. Load and normalize all summaries
# # ============================================================

# all_dfs = []
# for f in summary_files:
#     try:
#         df = pd.read_csv(f)
#         df = normalize_summary_df(df, f)
#         all_dfs.append(df)
#     except Exception as e:
#         print(f"Skipping {f} due to error: {e}")

# if not all_dfs:
#     raise ValueError("No summary files could be loaded.")

# results_df = pd.concat(all_dfs, ignore_index=True)


# # ============================================================
# # 6. Clean numeric types
# # ============================================================

# numeric_cols = [
#     "rows",
#     "seconds",
#     "seconds_per_item",
#     "comet",
#     "max_new_tokens",
#     "comet_batch_size",
#     "original_total_bytes",
#     "compressed_total_bytes",
#     "compression_ratio",
#     "comet_diff_vs_baseline",
#     "speedup_vs_baseline",
#     "current_total_bytes",
#     "model_size_on_disk_gb",
#     "compression_ratio_disk",
# ]

# for col in numeric_cols:
#     if col in results_df.columns:
#         results_df[col] = pd.to_numeric(results_df[col], errors="coerce")


# # ============================================================
# # 7. Drop duplicates
# # ============================================================

# results_df = results_df.drop_duplicates(
#     subset=["experiment_name", "run", "split", "rows", "seconds", "comet"],
#     keep="first",
# ).reset_index(drop=True)


# # ============================================================
# # 8. Infer cleaner method labels
# # ============================================================

# def infer_method_label(row) -> str:
#     run = str(row.get("run", "")).lower()
#     qmethod = str(row.get("quantization_method", "")).lower()
#     exp = str(row.get("experiment_name", "")).lower()

#     if qmethod == "none" or "baseline_fp16" in run or "fp16" in run:
#         return "none"
#     if "bitsandbytes" in qmethod or "int8" in run or "int8" in qmethod:
#         return "bitsandbytes_int8"
#     if "global" in run or "global" in exp:
#         return "global_codec"
#     if "structured_pruning" in run or "structured_pruning" in exp:
#         if bool(row.get("apply_codec_after_pruning", False)):
#             return "selected_layer_structured_pruning_plus_codec"
#         return "selected_layer_structured_pruning"
#     if "selected_layer_codec" in run or "selected_linear_codec" in run or "selected_linear" in exp:
#         return "custom_selected_linear_codec"
#     if qmethod not in {"", "nan", "unknown"}:
#         return str(row.get("quantization_method"))
#     return "unknown"

# results_df["method_label"] = results_df.apply(infer_method_label, axis=1)


# # ============================================================
# # 9. Add COMET delta vs baseline per split
# # ============================================================

# baseline_mask = (
#     results_df["quantization_method"].fillna("").eq("none")
#     | results_df["precision"].fillna("").eq("fp16")
#     | results_df["method_label"].eq("none")
# )

# baseline_by_split = (
#     results_df[baseline_mask]
#     .groupby("split", dropna=False)["comet"]
#     .max()
#     .to_dict()
# )

# if not baseline_by_split:
#     print("warning: no fp16 baseline rows were found, so Δ COMET will remain NaN")

# results_df["baseline_comet_for_split"] = results_df["split"].map(baseline_by_split)
# results_df["comet_delta_vs_baseline"] = results_df["comet"] - results_df["baseline_comet_for_split"]
# results_df["relative_comet_drop_pct_vs_baseline"] = (
#     (results_df["comet"] - results_df["baseline_comet_for_split"]) /
#     results_df["baseline_comet_for_split"]
# ) * 100.0


# # ============================================================
# # 10. Infer baseline original bytes per model
# # ============================================================

# baseline_original_bytes_by_model = (
#     results_df.groupby("model_id", dropna=False)["original_total_bytes"]
#     .max()
#     .to_dict()
# )


# def fill_original_bytes(row):
#     val = row.get("original_total_bytes", np.nan)
#     if pd.notna(val):
#         return val
#     return baseline_original_bytes_by_model.get(row.get("model_id"), np.nan)

# results_df["original_total_bytes_filled"] = results_df.apply(fill_original_bytes, axis=1)


# # ============================================================
# # 11. Compute effective current/compressed bytes
# # ============================================================

# def compute_effective_current_bytes(row):
#     compressed = row.get("compressed_total_bytes", np.nan)
#     current = row.get("current_total_bytes", np.nan)
#     original = row.get("original_total_bytes_filled", np.nan)
#     method = row.get("method_label", "")

#     if pd.notna(compressed):
#         return compressed
#     if method == "none" and pd.notna(original):
#         return original
#     if pd.notna(current):
#         return current
#     return np.nan

# results_df["effective_current_total_bytes"] = results_df.apply(compute_effective_current_bytes, axis=1)


# # ============================================================
# # 12. Recompute compression ratio consistently when possible
# # ============================================================

# def compute_effective_ratio(row):
#     original = row.get("original_total_bytes_filled", np.nan)
#     current = row.get("effective_current_total_bytes", np.nan)
#     if pd.notna(original) and pd.notna(current) and current > 0:
#         return original / current
#     return np.nan

# results_df["effective_compression_ratio"] = results_df.apply(compute_effective_ratio, axis=1)


# # ============================================================
# # 13. Add GB columns from the filled/effective sizes
# # ============================================================

# results_df["original_total_gb_filled"] = results_df["original_total_bytes_filled"] / BYTES_IN_GB
# results_df["effective_current_total_gb"] = results_df["effective_current_total_bytes"] / BYTES_IN_GB


# # ============================================================
# # 14. Sort full table
# # ============================================================

# results_df = results_df.sort_values(
#     by=["split", "comet", "seconds_per_item"],
#     ascending=[True, False, True],
# ).reset_index(drop=True)

# print("\nFull aggregated table:\n")
# display(results_df)


# # ============================================================
# # 15. Best per experiment/run/split
# # ============================================================

# best_per_run_split = (
#     results_df.sort_values(by=["comet", "seconds_per_item"], ascending=[False, True])
#     .drop_duplicates(subset=["experiment_name", "run", "split"], keep="first")
#     .reset_index(drop=True)
# )

# print("\nBest per experiment/run/split:\n")
# display(best_per_run_split)


# # ============================================================
# # 16. Best per split
# # ============================================================

# best_per_split = (
#     results_df.sort_values(by=["comet", "seconds_per_item"], ascending=[False, True])
#     .drop_duplicates(subset=["split"], keep="first")
#     .reset_index(drop=True)
# )

# print("\nBest per split:\n")
# display(best_per_split)


# # ============================================================
# # 17. Paper-friendly table
# # ============================================================

# paper_table = results_df[
#     [
#         "experiment_name",
#         "run",
#         "split",
#         "method_label",
#         "precision",
#         "rows",
#         "comet",
#         "comet_delta_vs_baseline",
#         "relative_comet_drop_pct_vs_baseline",
#         "seconds",
#         "seconds_per_item",
#         "effective_compression_ratio",
#         "original_total_gb_filled",
#         "effective_current_total_gb",
#         "max_new_tokens",
#     ]
# ].copy()

# paper_table["comet"] = paper_table["comet"].round(6)
# paper_table["comet_delta_vs_baseline"] = paper_table["comet_delta_vs_baseline"].round(6)
# paper_table["relative_comet_drop_pct_vs_baseline"] = paper_table["relative_comet_drop_pct_vs_baseline"].round(3)
# paper_table["seconds"] = paper_table["seconds"].round(3)
# paper_table["seconds_per_item"] = paper_table["seconds_per_item"].round(6)
# paper_table["effective_compression_ratio"] = paper_table["effective_compression_ratio"].round(3)
# paper_table["original_total_gb_filled"] = paper_table["original_total_gb_filled"].round(3)
# paper_table["effective_current_total_gb"] = paper_table["effective_current_total_gb"].round(3)

# paper_table = paper_table.sort_values(
#     by=["split", "comet", "seconds_per_item"],
#     ascending=[True, False, True],
# ).reset_index(drop=True)

# print("\nPaper-friendly table:\n")
# display(paper_table)


# # ============================================================
# # 18. Compact paper table
# # ============================================================

# compact_paper_table = paper_table.copy()
# compact_paper_table = compact_paper_table.rename(
#     columns={
#         "experiment_name": "Experiment",
#         "method_label": "Method",
#         "precision": "Prec.",
#         "comet": "COMET",
#         "comet_delta_vs_baseline": "Δ COMET",
#         "seconds_per_item": "Sec/item",
#         "effective_compression_ratio": "Comp. Ratio",
#         "original_total_gb_filled": "Orig GB",
#         "effective_current_total_gb": "Comp GB",
#     }
# )

# compact_paper_table = compact_paper_table[
#     [
#         "Experiment",
#         "Method",
#         "Prec.",
#         "COMET",
#         "Δ COMET",
#         "Sec/item",
#         "Comp. Ratio",
#         "Orig GB",
#         "Comp GB",
#     ]
# ].copy()

# compact_paper_table = compact_paper_table.sort_values(
#     by=["COMET", "Sec/item"],
#     ascending=[False, True],
# ).reset_index(drop=True)

# display_table = compact_paper_table.where(pd.notna(compact_paper_table), "N/A")

# print("\nCompact paper table:\n")
# display(display_table)


# # ============================================================
# # 19. Save outputs
# # ============================================================

# full_table_csv = REPORTS_DIR / "all_experiment_results_table.csv"
# best_run_split_csv = REPORTS_DIR / "best_per_experiment_run_split.csv"
# best_split_csv = REPORTS_DIR / "best_per_split.csv"
# paper_table_csv = REPORTS_DIR / "paper_results_table.csv"
# compact_paper_table_csv = REPORTS_DIR / "paper_results_table_compact.csv"
# compact_paper_table_display_csv = REPORTS_DIR / "paper_results_table_compact_display.csv"

# results_df.to_csv(full_table_csv, index=False)
# best_per_run_split.to_csv(best_run_split_csv, index=False)
# best_per_split.to_csv(best_split_csv, index=False)
# paper_table.to_csv(paper_table_csv, index=False)
# compact_paper_table.to_csv(compact_paper_table_csv, index=False)
# display_table.to_csv(compact_paper_table_display_csv, index=False)

# print("\nSaved:")
# print("-", full_table_csv)
# print("-", best_run_split_csv)
# print("-", best_split_csv)
# print("-", paper_table_csv)
# print("-", compact_paper_table_csv)
# print("-", compact_paper_table_display_csv)


In [2]:
# # ============================================================
# # OPTIONAL DELTA COMET VS FP16 BASELINE
# # ============================================================
# import os
# BASELINE_SUMMARY_CSV = os.environ.get(
#     "BASELINE_SUMMARY_CSV",
#     str(PROJECT_ROOT / "outputs" / "experiment_results" / "qwen2audio_eval_only_en_zh_bnb_int8_maxnew64" / "summary_comet_only.csv")
# )

# baseline_comet_for_split = np.nan
# comet_delta_vs_baseline = np.nan

# baseline_path = Path(BASELINE_SUMMARY_CSV)
# if baseline_path.exists():
#     baseline_df = pd.read_csv(baseline_path)

#     # keep only fp16 baseline rows if present
#     if "precision" in baseline_df.columns:
#         baseline_df_fp16 = baseline_df[baseline_df["precision"].astype(str).str.lower() == "fp16"].copy()
#         if not baseline_df_fp16.empty:
#             baseline_df = baseline_df_fp16

#     # try to match by split first
#     if "split" in baseline_df.columns:
#         baseline_df_split = baseline_df[baseline_df["split"].astype(str) == "eval"].copy()
#         if not baseline_df_split.empty:
#             baseline_df = baseline_df_split

#     # try to match by row count too, if available
#     if "rows" in baseline_df.columns:
#         baseline_df_rows = baseline_df[baseline_df["rows"] == len(pred_df)].copy()
#         if not baseline_df_rows.empty:
#             baseline_df = baseline_df_rows

#     # extract baseline COMET
#     if "comet" in baseline_df.columns and not baseline_df.empty:
#         baseline_comet_for_split = float(baseline_df.iloc[0]["comet"])
#         comet_delta_vs_baseline = float(comet_score - baseline_comet_for_split)
#         print("baseline summary used:", baseline_path)
#         print("baseline COMET:", baseline_comet_for_split)
#         print("Δ COMET:", comet_delta_vs_baseline)
#     else:
#         print("warning: baseline summary found but no usable fp16 COMET row:", baseline_path)
# else:
#     print("warning: baseline summary CSV not found:", baseline_path)

In [3]:
# from pathlib import Path

# root = Path("/home/paperspace/projects/iwslt2026-compression/outputs")
# matches = sorted(root.rglob("*qwen2audio*summary*.csv"))

# print("found", len(matches), "candidate baseline summary files\n")
# for p in matches:
#     print(p)

In [4]:
# from pathlib import Path
# import pandas as pd
# import numpy as np

# try:
#     from IPython.display import display
# except Exception:
#     def display(x):
#         print(x)

# # ============================================================
# # Aggregate all experiment summary spreadsheets
# # Fixed full script for notebook/script use
# # ============================================================

# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
# REPORTS_DIR = OUTPUTS_ROOT / "reports"
# REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# BYTES_IN_GB = 1024 ** 3


# # ============================================================
# # 1. Find summary files
# #    Search both live outputs and outputs-backup
# #    Ignore summary_experiment.csv to avoid duplicate rows
# # ============================================================

# summary_roots = [
#     PROJECT_ROOT / "outputs" / "experiment_results",
#     PROJECT_ROOT / "outputs-backup" / "experiment_results",
# ]

# summary_files = []
# for root in summary_roots:
#     if root.exists():
#         summary_files.extend(
#             f for f in root.glob("**/*summary*.csv")
#             if f.name != "summary_experiment.csv"
#         )

# summary_files = sorted(set(summary_files))

# print("Found summary files:")
# for f in summary_files:
#     print("-", f)

# if not summary_files:
#     raise ValueError("No summary files found.")


# # ============================================================
# # 2. Target schema
# # ============================================================

# base_columns = [
#     "timestamp_utc",
#     "experiment_name",
#     "run",
#     "split",
#     "rows",
#     "seconds",
#     "seconds_per_item",
#     "comet",
#     "model_id",
#     "prompt_text",
#     "max_new_tokens",
#     "comet_batch_size",
#     "predictions_csv",
#     "quantization_method",
#     "precision",
#     "eval_split_source",
# ]

# extra_columns = [
#     "original_total_bytes",
#     "compressed_total_bytes",
#     "compression_ratio",
#     "compression_method",
#     "comet_diff_vs_baseline",
#     "speedup_vs_baseline",
#     "current_total_bytes",
#     "size_scope",
#     "model_size_on_disk_gb",
#     "compression_ratio_disk",
#     "apply_codec_after_pruning",
# ]

# all_target_columns = base_columns + extra_columns


# # ============================================================
# # 3. Helpers
# # ============================================================

# def first_present(df: pd.DataFrame, candidates, default=np.nan):
#     for c in candidates:
#         if c in df.columns:
#             return df[c]
#     return pd.Series([default] * len(df), index=df.index)


# def infer_precision(df: pd.DataFrame) -> pd.Series:
#     run_series = df["run"].astype(str).str.lower() if "run" in df.columns else pd.Series([""] * len(df), index=df.index)
#     q_series = df["quantization_method"].astype(str).str.lower() if "quantization_method" in df.columns else pd.Series([""] * len(df), index=df.index)

#     precision = pd.Series(["custom"] * len(df), index=df.index)
#     precision[(run_series.str.contains("fp16")) | (q_series.eq("none"))] = "fp16"
#     precision[(run_series.str.contains("int8")) | (q_series.str.contains("int8")) | (q_series.str.contains("bitsandbytes"))] = "int8"
#     return precision


# # ============================================================
# # 4. Normalize one dataframe
# # ============================================================

# def normalize_summary_df(df: pd.DataFrame, file_path: Path) -> pd.DataFrame:
#     df = df.copy()
#     df["summary_csv"] = str(file_path)

#     if "timestamp_utc" not in df.columns:
#         df["timestamp_utc"] = pd.NaT

#     if "experiment_name" not in df.columns:
#         df["experiment_name"] = file_path.parent.name

#     if "split" not in df.columns:
#         df["split"] = "eval"

#     if "seconds_per_item" not in df.columns:
#         if "seconds" in df.columns and "rows" in df.columns:
#             df["seconds_per_item"] = df["seconds"] / df["rows"]
#         else:
#             df["seconds_per_item"] = np.nan

#     if "quantization_method" not in df.columns:
#         if "compression_method" in df.columns:
#             df["quantization_method"] = df["compression_method"]
#         else:
#             df["quantization_method"] = "unknown"

#     if "precision" not in df.columns:
#         df["precision"] = infer_precision(df)

#     if "prompt_text" not in df.columns:
#         df["prompt_text"] = np.nan

#     if "comet_batch_size" not in df.columns:
#         df["comet_batch_size"] = np.nan

#     if "predictions_csv" not in df.columns:
#         df["predictions_csv"] = np.nan

#     if "eval_split_source" not in df.columns:
#         df["eval_split_source"] = df["split"]

#     if "original_total_bytes" not in df.columns:
#         df["original_total_bytes"] = first_present(df, ["original_total_linear_bytes_est", "original_total_bytes"])

#     if "compressed_total_bytes" not in df.columns:
#         df["compressed_total_bytes"] = first_present(df, ["compressed_total_linear_bytes_est", "compressed_total_bytes"])

#     if "current_total_bytes" not in df.columns:
#         df["current_total_bytes"] = first_present(df, ["current_total_linear_bytes_est", "current_total_bytes"])

#     if "compression_ratio" not in df.columns:
#         df["compression_ratio"] = first_present(df, ["compression_ratio_est", "compression_ratio"])

#     if "size_scope" not in df.columns:
#         df["size_scope"] = first_present(df, ["size_scope"], default=np.nan)

#     if "model_size_on_disk_gb" not in df.columns:
#         df["model_size_on_disk_gb"] = first_present(df, ["model_size_on_disk_gb"], default=np.nan)

#     if "compression_ratio_disk" not in df.columns:
#         df["compression_ratio_disk"] = first_present(df, ["compression_ratio_disk"], default=np.nan)

#     if "apply_codec_after_pruning" not in df.columns:
#         df["apply_codec_after_pruning"] = False

#     for col in all_target_columns:
#         if col not in df.columns:
#             df[col] = np.nan

#     ordered_cols = all_target_columns + ["summary_csv"]
#     other_cols = [c for c in df.columns if c not in ordered_cols]
#     return df[ordered_cols + other_cols]


# # ============================================================
# # 5. Load and normalize all summaries
# # ============================================================

# all_dfs = []
# for f in summary_files:
#     try:
#         df = pd.read_csv(f)
#         df = normalize_summary_df(df, f)
#         all_dfs.append(df)
#     except Exception as e:
#         print(f"Skipping {f} due to error: {e}")

# if not all_dfs:
#     raise ValueError("No summary files could be loaded.")

# results_df = pd.concat(all_dfs, ignore_index=True)


# # ============================================================
# # 6. Clean numeric types
# # ============================================================

# numeric_cols = [
#     "rows",
#     "seconds",
#     "seconds_per_item",
#     "comet",
#     "max_new_tokens",
#     "comet_batch_size",
#     "original_total_bytes",
#     "compressed_total_bytes",
#     "compression_ratio",
#     "comet_diff_vs_baseline",
#     "speedup_vs_baseline",
#     "current_total_bytes",
#     "model_size_on_disk_gb",
#     "compression_ratio_disk",
# ]

# for col in numeric_cols:
#     if col in results_df.columns:
#         results_df[col] = pd.to_numeric(results_df[col], errors="coerce")


# # ============================================================
# # 7. Drop duplicates
# # ============================================================

# results_df = results_df.drop_duplicates(
#     subset=["experiment_name", "run", "split", "rows", "seconds", "comet"],
#     keep="first",
# ).reset_index(drop=True)


# # ============================================================
# # 8. Infer cleaner method labels
# # ============================================================

# def infer_method_label(row) -> str:
#     run = str(row.get("run", "")).lower()
#     qmethod = str(row.get("quantization_method", "")).lower()
#     exp = str(row.get("experiment_name", "")).lower()

#     if qmethod == "none" or "baseline_fp16" in run or "fp16" in run:
#         return "none"
#     if "bitsandbytes" in qmethod or "int8" in run or "int8" in qmethod:
#         return "bitsandbytes_int8"
#     if "global" in run or "global" in exp:
#         return "global_codec"
#     if "structured_pruning" in run or "structured_pruning" in exp:
#         if bool(row.get("apply_codec_after_pruning", False)):
#             return "selected_layer_structured_pruning_plus_codec"
#         return "selected_layer_structured_pruning"
#     if "selected_layer_codec" in run or "selected_linear_codec" in run or "selected_linear" in exp:
#         return "custom_selected_linear_codec"
#     if qmethod not in {"", "nan", "unknown"}:
#         return str(row.get("quantization_method"))
#     return "unknown"

# results_df["method_label"] = results_df.apply(infer_method_label, axis=1)


# # ============================================================
# # 9. Add COMET delta vs baseline per split and row count
# # ============================================================

# baseline_mask = (
#     results_df["quantization_method"].fillna("").eq("none")
#     | results_df["precision"].fillna("").eq("fp16")
#     | results_df["method_label"].eq("none")
# )

# baseline_rows = results_df[baseline_mask].copy()

# if baseline_rows.empty:
#     print("warning: no fp16 baseline rows were found, so Δ COMET will remain NaN")
#     results_df["baseline_comet_for_split"] = np.nan
#     results_df["comet_delta_vs_baseline"] = np.nan
#     results_df["relative_comet_drop_pct_vs_baseline"] = np.nan
# else:
#     baseline_lookup = (
#         baseline_rows
#         .groupby(["split", "rows"], dropna=False)["comet"]
#         .max()
#         .to_dict()
#     )

#     results_df["baseline_comet_for_split"] = results_df.apply(
#         lambda r: baseline_lookup.get((r["split"], r["rows"]), np.nan),
#         axis=1,
#     )
#     results_df["comet_delta_vs_baseline"] = (
#         results_df["comet"] - results_df["baseline_comet_for_split"]
#     )
#     results_df["relative_comet_drop_pct_vs_baseline"] = (
#         results_df["comet_delta_vs_baseline"] / results_df["baseline_comet_for_split"]
#     ) * 100.0


# # ============================================================
# # 10. Infer baseline original bytes per model
# # ============================================================

# baseline_original_bytes_by_model = (
#     results_df.groupby("model_id", dropna=False)["original_total_bytes"]
#     .max()
#     .to_dict()
# )


# def fill_original_bytes(row):
#     val = row.get("original_total_bytes", np.nan)
#     if pd.notna(val):
#         return val
#     return baseline_original_bytes_by_model.get(row.get("model_id"), np.nan)

# results_df["original_total_bytes_filled"] = results_df.apply(fill_original_bytes, axis=1)


# # ============================================================
# # 11. Compute effective current/compressed bytes
# # ============================================================

# def compute_effective_current_bytes(row):
#     compressed = row.get("compressed_total_bytes", np.nan)
#     current = row.get("current_total_bytes", np.nan)
#     original = row.get("original_total_bytes_filled", np.nan)
#     method = row.get("method_label", "")

#     if pd.notna(compressed):
#         return compressed
#     if method == "none" and pd.notna(original):
#         return original
#     if pd.notna(current):
#         return current
#     return np.nan

# results_df["effective_current_total_bytes"] = results_df.apply(compute_effective_current_bytes, axis=1)


# # ============================================================
# # 12. Recompute compression ratio consistently when possible
# # ============================================================

# def compute_effective_ratio(row):
#     original = row.get("original_total_bytes_filled", np.nan)
#     current = row.get("effective_current_total_bytes", np.nan)
#     if pd.notna(original) and pd.notna(current) and current > 0:
#         return original / current
#     return np.nan

# results_df["effective_compression_ratio"] = results_df.apply(compute_effective_ratio, axis=1)


# # ============================================================
# # 13. Add GB columns from the filled/effective sizes
# # ============================================================

# results_df["original_total_gb_filled"] = results_df["original_total_bytes_filled"] / BYTES_IN_GB
# results_df["effective_current_total_gb"] = results_df["effective_current_total_bytes"] / BYTES_IN_GB


# # ============================================================
# # 14. Sort full table
# # ============================================================

# results_df = results_df.sort_values(
#     by=["split", "comet", "seconds_per_item"],
#     ascending=[True, False, True],
# ).reset_index(drop=True)

# print("\nFull aggregated table:\n")
# display(results_df)


# # ============================================================
# # 15. Best per experiment/run/split
# # ============================================================

# best_per_run_split = (
#     results_df.sort_values(by=["comet", "seconds_per_item"], ascending=[False, True])
#     .drop_duplicates(subset=["experiment_name", "run", "split"], keep="first")
#     .reset_index(drop=True)
# )

# print("\nBest per experiment/run/split:\n")
# display(best_per_run_split)


# # ============================================================
# # 16. Best per split
# # ============================================================

# best_per_split = (
#     results_df.sort_values(by=["comet", "seconds_per_item"], ascending=[False, True])
#     .drop_duplicates(subset=["split"], keep="first")
#     .reset_index(drop=True)
# )

# print("\nBest per split:\n")
# display(best_per_split)


# # ============================================================
# # 17. Paper-friendly table
# # ============================================================

# paper_table = results_df[
#     [
#         "experiment_name",
#         "run",
#         "split",
#         "method_label",
#         "precision",
#         "rows",
#         "comet",
#         "comet_delta_vs_baseline",
#         "relative_comet_drop_pct_vs_baseline",
#         "seconds",
#         "seconds_per_item",
#         "effective_compression_ratio",
#         "original_total_gb_filled",
#         "effective_current_total_gb",
#         "max_new_tokens",
#     ]
# ].copy()

# paper_table["comet"] = paper_table["comet"].round(6)
# paper_table["comet_delta_vs_baseline"] = paper_table["comet_delta_vs_baseline"].round(6)
# paper_table["relative_comet_drop_pct_vs_baseline"] = paper_table["relative_comet_drop_pct_vs_baseline"].round(3)
# paper_table["seconds"] = paper_table["seconds"].round(3)
# paper_table["seconds_per_item"] = paper_table["seconds_per_item"].round(6)
# paper_table["effective_compression_ratio"] = paper_table["effective_compression_ratio"].round(3)
# paper_table["original_total_gb_filled"] = paper_table["original_total_gb_filled"].round(3)
# paper_table["effective_current_total_gb"] = paper_table["effective_current_total_gb"].round(3)

# paper_table = paper_table.sort_values(
#     by=["split", "comet", "seconds_per_item"],
#     ascending=[True, False, True],
# ).reset_index(drop=True)

# print("\nPaper-friendly table:\n")
# display(paper_table)


# # ============================================================
# # 18. Compact paper table
# # ============================================================

# compact_paper_table = paper_table.copy()
# compact_paper_table = compact_paper_table.rename(
#     columns={
#         "experiment_name": "Experiment",
#         "method_label": "Method",
#         "precision": "Prec.",
#         "comet": "COMET",
#         "comet_delta_vs_baseline": "Δ COMET",
#         "seconds_per_item": "Sec/item",
#         "effective_compression_ratio": "Comp. Ratio",
#         "original_total_gb_filled": "Orig GB",
#         "effective_current_total_gb": "Comp GB",
#     }
# )

# compact_paper_table = compact_paper_table[
#     [
#         "Experiment",
#         "Method",
#         "Prec.",
#         "COMET",
#         "Δ COMET",
#         "Sec/item",
#         "Comp. Ratio",
#         "Orig GB",
#         "Comp GB",
#     ]
# ].copy()

# compact_paper_table = compact_paper_table.sort_values(
#     by=["COMET", "Sec/item"],
#     ascending=[False, True],
# ).reset_index(drop=True)

# display_table = compact_paper_table.where(pd.notna(compact_paper_table), "N/A")

# print("\nCompact paper table:\n")
# display(display_table)


# # ============================================================
# # 19. Save outputs
# # ============================================================

# full_table_csv = REPORTS_DIR / "all_experiment_results_table.csv"
# best_run_split_csv = REPORTS_DIR / "best_per_experiment_run_split.csv"
# best_split_csv = REPORTS_DIR / "best_per_split.csv"
# paper_table_csv = REPORTS_DIR / "paper_results_table.csv"
# compact_paper_table_csv = REPORTS_DIR / "paper_results_table_compact.csv"
# compact_paper_table_display_csv = REPORTS_DIR / "paper_results_table_compact_display.csv"

# results_df.to_csv(full_table_csv, index=False)
# best_per_run_split.to_csv(best_run_split_csv, index=False)
# best_per_split.to_csv(best_split_csv, index=False)
# paper_table.to_csv(paper_table_csv, index=False)
# compact_paper_table.to_csv(compact_paper_table_csv, index=False)
# display_table.to_csv(compact_paper_table_display_csv, index=False)

# print("\nSaved:")
# print("-", full_table_csv)
# print("-", best_run_split_csv)
# print("-", best_split_csv)
# print("-", paper_table_csv)
# print("-", compact_paper_table_csv)
# print("-", compact_paper_table_display_csv)


In [8]:

# from pathlib import Path
# import numpy as np
# import pandas as pd

# PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
# LIVE_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"
# BACKUP_ROOT = PROJECT_ROOT / "outputs-backup" / "experiment_results"
# REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
# REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# BASELINE_MODEL_GB = 14.436


# def discover_summary_files():
#     files = []
#     for root in [LIVE_ROOT, BACKUP_ROOT]:
#         if root.exists():
#             files.extend(
#                 f for f in root.glob("**/*summary*.csv")
#                 if f.name != "summary_experiment.csv"
#             )
#     files = sorted(set(files))
#     if not files:
#         raise ValueError("No summary files found.")
#     return files


# def load_rows(summary_files):
#     frames = []
#     for path in summary_files:
#         try:
#             df = pd.read_csv(path)
#         except Exception as e:
#             print(f"warning: failed reading {path}: {e}")
#             continue
#         if df.empty:
#             continue
#         df = df.copy()
#         df["summary_csv_path"] = str(path)
#         df["summary_source"] = "live" if str(path).startswith(str(LIVE_ROOT)) else "backup"
#         frames.append(df)
#     if not frames:
#         raise ValueError("No readable summary CSVs found.")
#     return pd.concat(frames, ignore_index=True, sort=False)


# def normalize_experiment_name(row):
#     for c in ["experiment_name", "run"]:
#         if c in row.index and pd.notna(row[c]):
#             return str(row[c])
#     return "unknown_experiment"


# def normalize_method(row):
#     qm = str(row.get("quantization_method", "")).strip().lower()
#     exp = str(row.get("experiment_name", "")).strip().lower()
#     run = str(row.get("run", "")).strip().lower()
#     combo = " ".join([qm, exp, run])

#     if qm == "none" or str(row.get("precision", "")).strip().lower() == "fp16" or "baseline_fp16" in combo:
#         return "fp16_baseline"
#     if qm == "bitsandbytes_int8" or "quantized_int8" in combo or "int8" in combo:
#         return "bitsandbytes_int8"
#     if "pruning" in combo:
#         return "selected_layer_structured_pruning_plus_codec"
#     if "global" in combo:
#         return "global_codec"
#     if qm in {"custom_selected_linear_codec", "selected_layer_codec"} or "selected_linear_codec" in combo or "selected_layer_codec" in combo:
#         return "selected_layer_codec"
#     return "unknown"


# def normalize_setting(row):
#     method = row["method_label"]
#     codec_digits = pd.to_numeric(row.get("codec_digits", np.nan), errors="coerce")
#     codec_zstd = pd.to_numeric(row.get("codec_zstd_level", np.nan), errors="coerce")
#     exp = str(row.get("experiment_name", "")).lower()

#     if method == "fp16_baseline":
#         return "fp16"
#     if method == "bitsandbytes_int8":
#         return "int8"
#     if method in {"selected_layer_codec", "global_codec"}:
#         if pd.notna(codec_digits):
#             if pd.notna(codec_zstd):
#                 return f"q{int(codec_digits)} + zstd-{int(codec_zstd)}"
#             return f"q{int(codec_digits)}"
#         if "q2" in exp:
#             return "q2 + zstd"
#         if "q3" in exp:
#             return "q3 + zstd-3"
#         return "codec"
#     if method == "selected_layer_structured_pruning_plus_codec":
#         if pd.notna(codec_digits):
#             return f"prune + q{int(codec_digits)}"
#         return "prune + codec"
#     return "N/A"


# def first_present_numeric(row, cols):
#     for c in cols:
#         if c in row.index and pd.notna(row[c]):
#             return pd.to_numeric(pd.Series([row[c]]), errors="coerce").iloc[0]
#     return np.nan


# def score_row_for_keep(row):
#     score = 0
#     if row.get("summary_source") == "live":
#         score += 100
#     if pd.notna(row.get("seconds_per_item_eff", np.nan)):
#         score += 10
#     if pd.notna(row.get("compressed_gb_eff", np.nan)):
#         score += 10
#     if pd.notna(row.get("compression_ratio_eff", np.nan)):
#         score += 5
#     if row.get("method_label") == "fp16_baseline":
#         score += 20
#     ratio = row.get("compression_ratio_eff", np.nan)
#     if row.get("method_label") != "fp16_baseline" and pd.notna(ratio) and abs(float(ratio) - 1.0) < 1e-9:
#         score -= 20
#     return score


# summary_files = discover_summary_files()
# results_df = load_rows(summary_files)

# results_df["experiment_name_norm"] = results_df.apply(normalize_experiment_name, axis=1)
# results_df["method_label"] = results_df.apply(normalize_method, axis=1)
# results_df["setting_label"] = results_df.apply(normalize_setting, axis=1)

# if "split" not in results_df.columns:
#     results_df["split"] = "eval"
# results_df["split"] = results_df["split"].fillna("eval")

# results_df["rows"] = pd.to_numeric(results_df.get("rows", np.nan), errors="coerce")
# results_df["comet"] = pd.to_numeric(results_df.get("comet", np.nan), errors="coerce")
# results_df["seconds_per_item_eff"] = pd.to_numeric(
#     results_df.get("seconds_per_item", results_df.get("seconds_per_example", np.nan)),
#     errors="coerce",
# )

# results_df["orig_bytes_eff"] = results_df.apply(
#     lambda r: first_present_numeric(r, ["original_total_bytes", "original_total_linear_bytes_est"]),
#     axis=1,
# )
# results_df["comp_bytes_eff"] = results_df.apply(
#     lambda r: first_present_numeric(r, ["compressed_total_bytes", "compressed_total_linear_bytes_est", "current_total_bytes", "compressed_bytes"]),
#     axis=1,
# )
# results_df["compression_ratio_eff"] = results_df.apply(
#     lambda r: first_present_numeric(r, ["compression_ratio", "compression_ratio_est", "total_ratio"]),
#     axis=1,
# )

# results_df["original_gb_eff"] = results_df["orig_bytes_eff"] / (1024 ** 3)
# results_df["compressed_gb_eff"] = results_df["comp_bytes_eff"] / (1024 ** 3)

# # Always display the same original full-model size for comparison.
# results_df["original_gb_display"] = float(BASELINE_MODEL_GB)

# # Force fp16 baseline full values before dedup.
# mask_fp16 = results_df["method_label"].eq("fp16_baseline")
# results_df.loc[mask_fp16, "original_gb_display"] = float(BASELINE_MODEL_GB)
# results_df.loc[mask_fp16, "compressed_gb_eff"] = float(BASELINE_MODEL_GB)
# results_df.loc[mask_fp16, "compression_ratio_eff"] = 1.0

# # Fill missing compressed GB from ratio where possible.
# mask_fill_comp = (
#     results_df["compressed_gb_eff"].isna()
#     & results_df["compression_ratio_eff"].notna()
#     & (results_df["compression_ratio_eff"] > 0)
# )
# results_df.loc[mask_fill_comp, "compressed_gb_eff"] = (
#     results_df.loc[mask_fill_comp, "original_gb_display"]
#     / results_df.loc[mask_fill_comp, "compression_ratio_eff"]
# )

# # Fill missing ratio from sizes where possible.
# mask_fill_ratio = (
#     results_df["compression_ratio_eff"].isna()
#     & results_df["compressed_gb_eff"].notna()
#     & (results_df["compressed_gb_eff"] > 0)
# )
# results_df.loc[mask_fill_ratio, "compression_ratio_eff"] = (
#     results_df.loc[mask_fill_ratio, "original_gb_display"]
#     / results_df.loc[mask_fill_ratio, "compressed_gb_eff"]
# )

# # BitsAndBytes: original size known, compressed size unknown unless explicitly measured.
# mask_bnb = results_df["method_label"].eq("bitsandbytes_int8")
# results_df.loc[mask_bnb, "original_gb_display"] = float(BASELINE_MODEL_GB)
# results_df.loc[mask_bnb & results_df["comp_bytes_eff"].isna(), "compressed_gb_eff"] = np.nan
# results_df.loc[mask_bnb & results_df["comp_bytes_eff"].isna(), "compression_ratio_eff"] = np.nan

# results_df["dedup_score"] = results_df.apply(score_row_for_keep, axis=1)

# dedup_keys = ["experiment_name_norm", "method_label", "setting_label", "split", "rows", "comet"]
# results_df = (
#     results_df.sort_values(by=["dedup_score"], ascending=False)
#     .drop_duplicates(subset=dedup_keys, keep="first")
#     .reset_index(drop=True)
# )

# # Re-apply critical fills AFTER dedup so they cannot be lost.
# mask_fp16 = results_df["method_label"].eq("fp16_baseline")
# mask_bnb = results_df["method_label"].eq("bitsandbytes_int8")

# results_df["original_gb_display"] = float(BASELINE_MODEL_GB)
# results_df.loc[mask_fp16, "compressed_gb_eff"] = float(BASELINE_MODEL_GB)
# results_df.loc[mask_fp16, "compression_ratio_eff"] = 1.0

# mask_fill_comp = (
#     results_df["compressed_gb_eff"].isna()
#     & results_df["compression_ratio_eff"].notna()
#     & (results_df["compression_ratio_eff"] > 0)
#     & ~mask_bnb
# )
# results_df.loc[mask_fill_comp, "compressed_gb_eff"] = (
#     results_df.loc[mask_fill_comp, "original_gb_display"]
#     / results_df.loc[mask_fill_comp, "compression_ratio_eff"]
# )

# baseline_rows = results_df[mask_fp16].copy()
# if baseline_rows.empty:
#     results_df["baseline_comet_for_split"] = np.nan
#     results_df["comet_delta_vs_baseline"] = np.nan
# else:
#     baseline_lookup = (
#         baseline_rows.groupby(["split", "rows"], dropna=False)["comet"].max().to_dict()
#     )
#     results_df["baseline_comet_for_split"] = results_df.apply(
#         lambda r: baseline_lookup.get((r["split"], r["rows"]), np.nan),
#         axis=1,
#     )
#     results_df["comet_delta_vs_baseline"] = results_df["comet"] - results_df["baseline_comet_for_split"]

# paper_table = pd.DataFrame({
#     "Experiment": results_df["experiment_name_norm"],
#     "Method": results_df["method_label"],
#     "Setting": results_df["setting_label"],
#     "COMET": results_df["comet"],
#     "Δ COMET": results_df["comet_delta_vs_baseline"],
#     "Sec/item": results_df["seconds_per_item_eff"],
#     "Comp. Ratio": results_df["compression_ratio_eff"],
#     "Orig GB": results_df["original_gb_display"],
#     "Comp GB": results_df["compressed_gb_eff"],
# })

# # Final hard override at display construction.
# paper_table["Orig GB"] = float(BASELINE_MODEL_GB)
# paper_table.loc[paper_table["Method"] == "fp16_baseline", "Comp GB"] = float(BASELINE_MODEL_GB)
# paper_table.loc[paper_table["Method"] == "fp16_baseline", "Comp. Ratio"] = 1.0

# method_order = {
#     "fp16_baseline": 0,
#     "bitsandbytes_int8": 1,
#     "selected_layer_codec": 2,
#     "global_codec": 3,
#     "selected_layer_structured_pruning_plus_codec": 4,
#     "unknown": 9,
# }
# paper_table["_method_order"] = paper_table["Method"].map(method_order).fillna(9)
# paper_table = (
#     paper_table.sort_values(by=["_method_order", "COMET", "Sec/item"], ascending=[True, False, True])
#     .drop(columns=["_method_order"])
#     .reset_index(drop=True)
# )

# for c, d in {
#     "COMET": 6,
#     "Δ COMET": 6,
#     "Sec/item": 6,
#     "Comp. Ratio": 3,
#     "Orig GB": 3,
#     "Comp GB": 3,
# }.items():
#     paper_table[c] = pd.to_numeric(paper_table[c], errors="coerce").round(d)

# display_table = paper_table.where(pd.notna(paper_table), "N/A")

# print(display_table.to_string(index=False))

# raw_debug_path = REPORTS_DIR / "paper_results_raw_debug.csv"
# clean_table_path = REPORTS_DIR / "paper_results_table_clean.csv"
# clean_display_path = REPORTS_DIR / "paper_results_table_clean_display.csv"

# results_df.to_csv(raw_debug_path, index=False)
# paper_table.to_csv(clean_table_path, index=False)
# display_table.to_csv(clean_display_path, index=False)

# print("\nSaved:")
# print("-", raw_debug_path)
# print("-", clean_table_path)
# print("-", clean_display_path)


In [9]:

from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
LIVE_ROOT = PROJECT_ROOT / "outputs" / "experiment_results"
BACKUP_ROOT = PROJECT_ROOT / "outputs-backup" / "experiment_results"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_MODEL_GB = 14.436


def discover_summary_files():
    files = []
    for root in [LIVE_ROOT, BACKUP_ROOT]:
        if root.exists():
            files.extend(
                f for f in root.glob("**/*summary*.csv")
                if f.name != "summary_experiment.csv"
            )
    files = sorted(set(files))
    if not files:
        raise ValueError("No summary files found.")
    return files


def load_rows(summary_files):
    frames = []
    for path in summary_files:
        try:
            df = pd.read_csv(path)
        except Exception as e:
            print(f"warning: failed reading {path}: {e}")
            continue
        if df.empty:
            continue
        df = df.copy()
        df["summary_csv_path"] = str(path)
        df["summary_source"] = "live" if str(path).startswith(str(LIVE_ROOT)) else "backup"
        frames.append(df)
    if not frames:
        raise ValueError("No readable summary CSVs found.")
    return pd.concat(frames, ignore_index=True, sort=False)


def normalize_experiment_name(row):
    for c in ["experiment_name", "run"]:
        if c in row.index and pd.notna(row[c]):
            return str(row[c])
    return "unknown_experiment"


def normalize_method(row):
    qm = str(row.get("quantization_method", "")).strip().lower()
    exp = str(row.get("experiment_name", "")).strip().lower()
    run = str(row.get("run", "")).strip().lower()
    combo = " ".join([qm, exp, run])

    if qm == "none" or str(row.get("precision", "")).strip().lower() == "fp16" or "baseline_fp16" in combo:
        return "fp16_baseline"
    if qm == "bitsandbytes_int8" or "quantized_int8" in combo or "int8" in combo:
        return "bitsandbytes_int8"
    if "pruning" in combo:
        return "selected_layer_structured_pruning_plus_codec"
    if "global" in combo:
        return "global_codec"
    if qm in {"custom_selected_linear_codec", "selected_layer_codec"} or "selected_linear_codec" in combo or "selected_layer_codec" in combo:
        return "selected_layer_codec"
    return "unknown"


def normalize_setting(row):
    method = row["method_label"]
    codec_digits = pd.to_numeric(row.get("codec_digits", np.nan), errors="coerce")
    codec_zstd = pd.to_numeric(row.get("codec_zstd_level", np.nan), errors="coerce")
    exp = str(row.get("experiment_name", "")).lower()

    if method == "fp16_baseline":
        return "fp16"
    if method == "bitsandbytes_int8":
        return "int8"
    if method in {"selected_layer_codec", "global_codec"}:
        if pd.notna(codec_digits):
            if pd.notna(codec_zstd):
                return f"q{int(codec_digits)} + zstd-{int(codec_zstd)}"
            return f"q{int(codec_digits)}"
        if "q2" in exp:
            return "q2 + zstd"
        if "q3" in exp:
            return "q3 + zstd-3"
        return "codec"
    if method == "selected_layer_structured_pruning_plus_codec":
        if pd.notna(codec_digits):
            return f"prune + q{int(codec_digits)}"
        return "prune + codec"
    return "N/A"


def first_present_numeric(row, cols):
    for c in cols:
        if c in row.index and pd.notna(row[c]):
            return pd.to_numeric(pd.Series([row[c]]), errors="coerce").iloc[0]
    return np.nan


def score_row_for_keep(row):
    score = 0
    if row.get("summary_source") == "live":
        score += 100
    if pd.notna(row.get("seconds_per_item_eff", np.nan)):
        score += 10
    if pd.notna(row.get("compressed_gb_eff", np.nan)):
        score += 10
    if pd.notna(row.get("compression_ratio_eff", np.nan)):
        score += 5
    if row.get("method_label") == "fp16_baseline":
        score += 20
    ratio = row.get("compression_ratio_eff", np.nan)
    if row.get("method_label") != "fp16_baseline" and pd.notna(ratio) and abs(float(ratio) - 1.0) < 1e-9:
        score -= 20
    return score


summary_files = discover_summary_files()
results_df = load_rows(summary_files)

results_df["experiment_name_norm"] = results_df.apply(normalize_experiment_name, axis=1)
results_df["method_label"] = results_df.apply(normalize_method, axis=1)
results_df["setting_label"] = results_df.apply(normalize_setting, axis=1)

if "split" not in results_df.columns:
    results_df["split"] = "eval"
results_df["split"] = results_df["split"].fillna("eval")

results_df["rows"] = pd.to_numeric(results_df.get("rows", np.nan), errors="coerce")
results_df["comet"] = pd.to_numeric(results_df.get("comet", np.nan), errors="coerce")

# Prefer an explicit per-item field, but derive it from seconds/rows when needed.
seconds_total = pd.to_numeric(results_df.get("seconds", np.nan), errors="coerce")
seconds_per_item_direct = pd.to_numeric(
    results_df.get("seconds_per_item", results_df.get("seconds_per_example", np.nan)),
    errors="coerce",
)
results_df["seconds_per_item_eff"] = seconds_per_item_direct
mask_missing_spi = (
    results_df["seconds_per_item_eff"].isna()
    & seconds_total.notna()
    & results_df["rows"].notna()
    & (results_df["rows"] > 0)
)
results_df.loc[mask_missing_spi, "seconds_per_item_eff"] = (
    seconds_total[mask_missing_spi] / results_df.loc[mask_missing_spi, "rows"]
)

results_df["orig_bytes_eff"] = results_df.apply(
    lambda r: first_present_numeric(r, ["original_total_bytes", "original_total_linear_bytes_est"]),
    axis=1,
)
results_df["comp_bytes_eff"] = results_df.apply(
    lambda r: first_present_numeric(r, ["compressed_total_bytes", "compressed_total_linear_bytes_est", "current_total_bytes", "compressed_bytes"]),
    axis=1,
)
results_df["compression_ratio_eff"] = results_df.apply(
    lambda r: first_present_numeric(r, ["compression_ratio", "compression_ratio_est", "total_ratio"]),
    axis=1,
)

results_df["original_gb_eff"] = results_df["orig_bytes_eff"] / (1024 ** 3)
results_df["compressed_gb_eff"] = results_df["comp_bytes_eff"] / (1024 ** 3)

results_df["original_gb_display"] = float(BASELINE_MODEL_GB)

mask_fp16 = results_df["method_label"].eq("fp16_baseline")
results_df.loc[mask_fp16, "original_gb_display"] = float(BASELINE_MODEL_GB)
results_df.loc[mask_fp16, "compressed_gb_eff"] = float(BASELINE_MODEL_GB)
results_df.loc[mask_fp16, "compression_ratio_eff"] = 1.0

mask_fill_comp = (
    results_df["compressed_gb_eff"].isna()
    & results_df["compression_ratio_eff"].notna()
    & (results_df["compression_ratio_eff"] > 0)
)
results_df.loc[mask_fill_comp, "compressed_gb_eff"] = (
    results_df.loc[mask_fill_comp, "original_gb_display"]
    / results_df.loc[mask_fill_comp, "compression_ratio_eff"]
)

mask_fill_ratio = (
    results_df["compression_ratio_eff"].isna()
    & results_df["compressed_gb_eff"].notna()
    & (results_df["compressed_gb_eff"] > 0)
)
results_df.loc[mask_fill_ratio, "compression_ratio_eff"] = (
    results_df.loc[mask_fill_ratio, "original_gb_display"]
    / results_df.loc[mask_fill_ratio, "compressed_gb_eff"]
)

mask_bnb = results_df["method_label"].eq("bitsandbytes_int8")
results_df.loc[mask_bnb, "original_gb_display"] = float(BASELINE_MODEL_GB)
results_df.loc[mask_bnb & results_df["comp_bytes_eff"].isna(), "compressed_gb_eff"] = np.nan
results_df.loc[mask_bnb & results_df["comp_bytes_eff"].isna(), "compression_ratio_eff"] = np.nan

results_df["dedup_score"] = results_df.apply(score_row_for_keep, axis=1)

dedup_keys = ["experiment_name_norm", "method_label", "setting_label", "split", "rows", "comet"]
results_df = (
    results_df.sort_values(by=["dedup_score"], ascending=False)
    .drop_duplicates(subset=dedup_keys, keep="first")
    .reset_index(drop=True)
)

mask_fp16 = results_df["method_label"].eq("fp16_baseline")
mask_bnb = results_df["method_label"].eq("bitsandbytes_int8")

results_df["original_gb_display"] = float(BASELINE_MODEL_GB)
results_df.loc[mask_fp16, "compressed_gb_eff"] = float(BASELINE_MODEL_GB)
results_df.loc[mask_fp16, "compression_ratio_eff"] = 1.0

mask_fill_comp = (
    results_df["compressed_gb_eff"].isna()
    & results_df["compression_ratio_eff"].notna()
    & (results_df["compression_ratio_eff"] > 0)
    & ~mask_bnb
)
results_df.loc[mask_fill_comp, "compressed_gb_eff"] = (
    results_df.loc[mask_fill_comp, "original_gb_display"]
    / results_df.loc[mask_fill_comp, "compression_ratio_eff"]
)

baseline_rows = results_df[mask_fp16].copy()
if baseline_rows.empty:
    results_df["baseline_comet_for_split"] = np.nan
    results_df["comet_delta_vs_baseline"] = np.nan
else:
    baseline_lookup = (
        baseline_rows.groupby(["split", "rows"], dropna=False)["comet"].max().to_dict()
    )
    results_df["baseline_comet_for_split"] = results_df.apply(
        lambda r: baseline_lookup.get((r["split"], r["rows"]), np.nan),
        axis=1,
    )
    results_df["comet_delta_vs_baseline"] = results_df["comet"] - results_df["baseline_comet_for_split"]

paper_table = pd.DataFrame({
    "Experiment": results_df["experiment_name_norm"],
    "Method": results_df["method_label"],
    "Setting": results_df["setting_label"],
    "COMET": results_df["comet"],
    "Δ COMET": results_df["comet_delta_vs_baseline"],
    "Sec/item": results_df["seconds_per_item_eff"],
    "Comp. Ratio": results_df["compression_ratio_eff"],
    "Orig GB": results_df["original_gb_display"],
    "Comp GB": results_df["compressed_gb_eff"],
})

paper_table["Orig GB"] = float(BASELINE_MODEL_GB)
paper_table.loc[paper_table["Method"] == "fp16_baseline", "Comp GB"] = float(BASELINE_MODEL_GB)
paper_table.loc[paper_table["Method"] == "fp16_baseline", "Comp. Ratio"] = 1.0

method_order = {
    "fp16_baseline": 0,
    "bitsandbytes_int8": 1,
    "selected_layer_codec": 2,
    "global_codec": 3,
    "selected_layer_structured_pruning_plus_codec": 4,
    "unknown": 9,
}
paper_table["_method_order"] = paper_table["Method"].map(method_order).fillna(9)
paper_table = (
    paper_table.sort_values(by=["_method_order", "COMET", "Sec/item"], ascending=[True, False, True])
    .drop(columns=["_method_order"])
    .reset_index(drop=True)
)

for c, d in {
    "COMET": 6,
    "Δ COMET": 6,
    "Sec/item": 6,
    "Comp. Ratio": 3,
    "Orig GB": 3,
    "Comp GB": 3,
}.items():
    paper_table[c] = pd.to_numeric(paper_table[c], errors="coerce").round(d)

display_table = paper_table.where(pd.notna(paper_table), "N/A")

print(display_table.to_string(index=False))

raw_debug_path = REPORTS_DIR / "paper_results_raw_debug.csv"
clean_table_path = REPORTS_DIR / "paper_results_table_clean.csv"
clean_display_path = REPORTS_DIR / "paper_results_table_clean_display.csv"

results_df.to_csv(raw_debug_path, index=False)
paper_table.to_csv(clean_table_path, index=False)
display_table.to_csv(clean_display_path, index=False)

print("\nSaved:")
print("-", raw_debug_path)
print("-", clean_table_path)
print("-", clean_display_path)


                                                       Experiment                                       Method     Setting    COMET   Δ COMET  Sec/item  Comp. Ratio  Orig GB  Comp GB
                     qwen2audio_eval_only_en_zh_bnb_int8_maxnew64                                fp16_baseline        fp16 0.779163  0.000000  0.664055        1.000   14.436   14.436
                     qwen2audio_eval_only_en_zh_bnb_int8_maxnew64                            bitsandbytes_int8        int8 0.775192 -0.003971  2.538633        1.733   14.436    9.026
custom_selected_linear_codec_all_selected_q3_zstd_eval_only_en_zh                         selected_layer_codec q3 + zstd-3 0.776689 -0.002473  0.751678        1.400   14.436   10.312
                     custom_codec_selected_linear_eval_only_en_zh                         selected_layer_codec       codec 0.776365 -0.002798  0.745540        1.714   14.436    8.422
                      custom_codec_global_q2_zstd_eval_only_en_zh                    

In [10]:
display_table

,Experiment,Method,Setting,COMET,Δ COMET,Sec/item,Comp. Ratio,Orig GB,Comp GB
0,qwen2audio_eval_only_en_zh_bnb_int8_maxnew64,fp16_baseline,fp16,0.779163,0.000000,0.664055,1.000,14.436,14.436
1,qwen2audio_eval_only_en_zh_bnb_int8_maxnew64,bitsandbytes_int8,int8,0.775192,-0.003971,2.538633,1.733,14.436,9.026
2,custom_selected_linear_codec_all_selected_q3_z...,selected_layer_codec,q3 + zstd-3,0.776689,-0.002473,0.751678,1.400,14.436,10.312
3,custom_codec_selected_linear_eval_only_en_zh,selected_layer_codec,codec,0.776365,-0.002798,0.745540,1.714,14.436,8.422
4,custom_codec_global_q2_zstd_eval_only_en_zh,global_codec,q2 + zstd,0.741091,-0.038071,0.836255,3.979,14.436,3.628
5,custom_selected_linear_structured_pruning_code...,selected_layer_structured_pruning_plus_codec,prune + q2,0.587115,-0.192048,0.856106,1.800,14.436,8.020
